In [3]:
#from google.colab import drive
#drive.mount('/content/drive')

# © 2026 Ionescu Andrei-Cristian, Ionescu Alex Gabriel, Voicu Daria Stefania. Toate drepturile rezervate. / All rights reserved.

🇷🇴 **ROMÂNĂ**

Acest cod sursă personalizat (inclusiv scripturile, logica de integrare, sistemul de filtrare a mișcării "Motion Gate" și dicționarul de mapare a emoțiilor) este proprietate privată și confidențială aparținând autorilor menționați mai sus.

*Notă: Modelele de detecție de bază (YOLO aparținând Ultralytics și Face Landmarker aparținând Google MediaPipe) sunt utilizate exclusiv ca instrumente terțe. Drepturile asupra acestora aparțin creatorilor originali, conform licențelor specifice de tip open-source.*

**Este strict interzisă:**
*   Copierea, distribuirea sau publicarea arhitecturii și logicii noastre de cod (parțial sau total) fără acordul explicit și scris al autorilor.
*   Utilizarea arhitecturii în scopuri comerciale, academice sau integrarea în alte proiecte derivate fără permisiune.
*   Modificarea și redistribuirea muncii noastre sub alt nume.

Orice utilizare neautorizată a logicii implementate de noi reprezintă o încălcare a drepturilor de autor.

---

🇬🇧 **ENGLISH**

This custom source code (including the scripts, integration logic, the "Motion Gate" filtering system, and the emotion mapping dictionary) is the private and confidential property of the aforementioned authors.

*Note: The core detection models (YOLO by Ultralytics and Face Landmarker by Google MediaPipe) are utilized strictly as third-party tools. All rights to these models belong to their original creators, governed by their respective open-source licenses.*

**The following actions are strictly prohibited:**
*   Copying, distributing, or publishing our custom code architecture and logic (in whole or in part) without the explicit written consent of the authors.
*   Using our architecture for commercial or academic purposes, or integrating it into other derivative projects without permission.
*   Modifying and redistributing our work under a different name.

Any unauthorized use of our implemented logic constitutes a violation of copyright laws.

In [2]:
!pip install ultralytics
!pip install Roboflow
!pip install mediapipe


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Punem drive-ul nostru pe serverul cu acest notebook pentru a putea salva ce lucram aici in drive
# !!! Toti trebuie sa ruleze acest cod !!!
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


instalare biblioteci


navigare fisiere


In [ ]:

%pwd


'/content'

Upload Dataset


In [ ]:
import ultralytics
import roboflow
import os
rf = roboflow.Roboflow(api_key = "OKfGZ3rpPvH45EuY1Nj2")
workspace = rf.workspace("bluefire64")

project = workspace.project("face_detection-uuyvs")

dataset_path = "Face_Detection.v1i.yolo26"
for filename in os.listdir(dataset_path):
  image_path = os.path.join(dataset_path, filename)
  if os.path.isfile(image_path) and filename.lower().endswith("jpg"):
    print(f"Uploading file {filename}...")
    project.upload(image_path)




Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
loading Roboflow workspace...
loading Roboflow project...


FileNotFoundError: [Errno 2] No such file or directory: 'Face_Detection.v1i.yolo26'

Configurare Antrenare YOLO



In [ ]:
import os
from ultralytics import YOLO

print("Fișierele și folderele găsite aici sunt:", os.listdir())

def train_yolo26_model():
    """
    Fine-tunes a YOLO26 model on a custom dataset exported from Roboflow.

    Prerequisites:
    1. pip install ultralytics
    2. Export your dataset from Roboflow in 'YOLO' format.
    3. Ensure the 'data.yaml' file provided by Roboflow has the correcta
       absolute or relative paths to your train/val image folders.
    """
    print("Initializing YOLO26 Model...")


    model = YOLO("yolo26n.pt")


    dataset_yaml_path = f"{dataset.location}/data.yaml"

    if not os.path.exists(dataset_yaml_path):
        print(f"Error: Could not find {dataset_yaml_path}.")
        print("Please ensure your Roboflow dataset is extracted in the 'Face_Detection.v1i.yolo26' folder.")
        return

    print(f"Starting training on dataset: {dataset_yaml_path}...")


    results = model.train(
        data=dataset_yaml_path,  # Path to the dataset config
        epochs=200,               # Number of training epochs
        imgsz=640,               # Resize images to 640x640 during training
        batch=16,                # Batch size (adjust based on your GPU memory)
        device="cuda",           # Use "cuda" for GPU, "cpu", or "mps" for Apple Silicon
        name="yolo26_faces_200_epoci",     # Name of the folder where results/weights will be saved


        hsv_h=0.015,             # Image HSV-Hue augmentation
        hsv_s=0.7,               # Image HSV-Saturation augmentation
        hsv_v=0.4,               # Image HSV-Value augmentation
        fliplr=0.5,              # 50% chance to flip images left-right (good for faces)
    )

    print("\nTraining Complete!")
    print(f"Best model weights saved to: runs/detect/yolo26_faces_200_epoci/weights/best.pt")


train_yolo26_model()

Fișierele și folderele găsite aici sunt: ['.config', 'drive', 'sample_data']
Initializing YOLO26 Model...


NameError: name 'dataset' is not defined

descarcare set de date rf

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="OKfGZ3rpPvH45EuY1Nj2")
project = rf.workspace("bluefire64").project("face_detection-uuyvs")
version = project.version(1)
dataset = version.download("yolov8")
print("Datele au fost descărcate cu succes!")

loading Roboflow workspace...
loading Roboflow project...
Datele au fost descărcate cu succes!


Antrenare propriu-zisa


In [ ]:
import os
from ultralytics import YOLO
from roboflow import Roboflow


print("Se descarcă datele din Roboflow...")

rf = Roboflow(api_key="OKfGZ3rpPvH45EuY1Nj2")
project = rf.workspace("bluefire64").project("face_detection-uuyvs")

version = project.version(1)
dataset = version.download("yolov8")


def train_yolo26_model():
    print("\nInitializing YOLO26 Model...")
    model = YOLO("yolo26n.pt")

    dataset_yaml_path = f"{dataset.location}/data.yaml"

    print(f"Starting training on dataset: {dataset_yaml_path}...")

    results = model.train(
        data=dataset_yaml_path,
        epochs=200,
        imgsz=640,
        batch=16,
        device="cuda",
        name="yolo26_faces_200_epoci",
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        fliplr=0.5,
    )

    print("\nTraining Complete!")
    print(f"Best model weights saved to: runs/detect/yolo26_faces_200_epoci/weights/best.pt")

train_yolo26_model()

Se descarcă datele din Roboflow...
loading Roboflow workspace...
loading Roboflow project...
Exporting format yolov8 in progress : 95.0%
Version export complete for yolov8 format



Extracting Dataset Version Zip to Face_Detection-1 in yolov8:: 100%|██████████| 2793/2793 [00:00<00:00, 6746.23it/s]



Initializing YOLO26 Model...
Starting training on dataset: /content/Face_Detection-1/data.yaml...
Ultralytics 8.4.108 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Face_Detection-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=

Inferenta Standard - testare model YOLO


In [ ]:
def run_inference_test():
    """
    Demonstrates how to load the fine-tuned model and run it on a new image.
    """
    best_model_path = "runs/detect/yolo26_faces/weights/best.pt"

    if os.path.exists(best_model_path):
        print("\nTesting the trained model...")
        custom_model = YOLO(best_model_path)

        for image in os.listdir("Face_Detection.v1i.yolo26/test/images"):
          if os.path.exists(os.path.join("Face_Detection.v1i.yolo26/test/images", image)):
            results = custom_model.predict(source=os.path.join("Face_Detection.v1i.yolo26/test/images", image), save=True, conf=0.5)
            print(f"Inference complete. Annotated image saved in the runs/detect/predict directory.")
          else:
            print(f"Test image not found at {os.path.join("Face_Detection.v1i.yolo26/test/images", image)}. Skipping inference test.")


run_inference_test()

Inferență Hibridă: YOLO Ajustat + MediaPipe

In [ ]:
import os
import cv2
from roboflow import Roboflow
from ultralytics import YOLO
import mediapipe as mp
import urllib.request
from collections import deque

# Thresholds

print("Verificam/Descarcam dataset-ul pentru a prelua calea...")
rf = Roboflow(api_key="OKfGZ3rpPvH45EuY1Nj2")
project = rf.workspace("bluefire64").project("face_detection-uuyvs")
version = project.version(1)
dataset = version.download("yolov8")
print(f"Dataset gasit la: {dataset.location}")

model_url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task"
urllib.request.urlretrieve(model_url, "face_landmarker.task")
print("Downloaded face_landmarker succesfully!")


def run_inference_test_blendshapes():
    best_model_path = "/content/drive/MyDrive/Team_Face_Dataset/runs/detect/yolo26_faces/weights/best.pt"

    if os.path.exists(best_model_path):
        print(f"\nModel găsit cu succes la: {best_model_path}")
        print("Marim STRICT latimea chenarelor...")
        custom_model = YOLO(best_model_path)


        test_images_path = f"{dataset.location}/test/images"
        test_images_path = "/content/drive/MyDrive/Team_Face_Dataset/Face_Detection.v1i.yolo26/test/images"
        output_dir = "/content/drive/MyDrive/Team_Face_Dataset/runs/detect/blendshapes"
        os.makedirs(output_dir, exist_ok=True)

        if not os.path.exists(test_images_path):
            print(f"Error: Nu găsesc folderul cu imagini de test la {test_images_path}")
            return
        BaseOptions = mp.tasks.BaseOptions
        FaceLandmarker = mp.tasks.vision.FaceLandmarker
        FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
        VisionRunningMode = mp.tasks.vision.RunningMode
        options = FaceLandmarkerOptions(
            base_options=BaseOptions(model_asset_path="face_landmarker.task"),
            running_mode=VisionRunningMode.IMAGE,
            output_face_blendshapes = True)
        with FaceLandmarker.create_from_options(options) as landmarker:
          for image in os.listdir(test_images_path):
              image_path = os.path.join(test_images_path, image)

              if os.path.exists(image_path) and image.lower().endswith(('.jpg', '.jpeg', '.png')):

                  results = custom_model.predict(source=image_path, save=False, conf=0.5)

                  img_cv2 = cv2.imread(image_path)
                  img_h, img_w = img_cv2.shape[:2]

                  for result in results:
                      boxes = result.boxes
                      for box in boxes:
                          x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()

                          w = x2 - x1
                          h = y2 - y1

                          pad_x = w * 0.1
                          x1_nou = max(0, int(x1 - pad_x))
                          x2_nou = min(img_w, int(x2 + pad_x))

                          y2_nou = int(y2)
                          y1_nou = int(y1)

                          raport = w / h
                          if raport > 0.9:
                              offset_frunte = h * 0.30
                              y1_nou = max(0, int(y1 - offset_frunte))

                              color = (0, 0, 255)
                              label = f"Ajustat sus + latit: W/H {raport:.2f}"
                          else:
                              color = (0, 255, 0)
                              label = f"Doar latit: W/H {raport:.2f}"

                          cv2.rectangle(img_cv2, (x1_nou, y1_nou), (x2_nou, y2_nou), color, 2)


                          text_y = max(20, y1_nou - 10)
                          cv2.putText(img_cv2, label, (x1_nou, text_y),
                                      cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

                          img_crop = img_cv2[y1_nou:y2_nou, x1_nou:x2_nou]
                          if img_crop.size == 0:
                            if img_crop.size == 0:
                              print(f"Eroare: crop cu dimensiune 0!\n"
                                    f"Path: {image_path}\n"
                                    f"Nume imagine: {image}")
                              continue

                          img_crop_rgb = cv2.cvtColor(img_crop, cv2.COLOR_BGR2RGB)
                          mp_img = mp.Image(image_format = mp.ImageFormat.SRGB, data = img_crop_rgb)

                          results = landmarker.detect(mp_img)

                          if results.face_landmarks:
                            landmarks = results.face_landmarks[0]
                            crop_h, crop_w, _ = img_crop.shape

                            for landmark in landmarks:
                              # Punem offset pentru a desena landmark-erii pe imaginea originala, nu pe cea cropped (e comentariul MEU nu al lui Gemini)
                              x = x1_nou + int(landmark.x * crop_w)
                              y = y1_nou + int(landmark.y * crop_h)
                              cv2.circle(img_cv2, (x, y), 1, (0, 255, 255), -1)

                          if results.face_blendshapes:
                            blendshapes = results.face_blendshapes[0]
                            for b in blendshapes:
                              if b.category_name in ["browInnerUp", "browDownLeft", "browDownRight", "mouthSmileLeft"]:
                                print(f"[{image}] {b.category_name} (AU Score): {b.score:.3f}")




                  save_path = os.path.join(output_dir, image)
                  cv2.imwrite(save_path, img_cv2)

          print(f"\nInference complete. Pozele au fost salvate in: {output_dir}")
    else:
        print(f"Eroare: Nu am găsit modelul la calea: {best_model_path}")

run_inference_test_blendshapes()

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Verificam/Descarcam dataset-ul pentru a prelua calea...
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Face_Detection-1 in yolov8:: 100%|██████████| 2793/2793 [00:00<00:00, 5912.12it/s]


Dataset gasit la: /content/Face_Detection-1
Downloaded face_landmarker succesfully!

Model găsit cu succes la: /content/drive/MyDrive/Team_Face_Dataset/runs/detect/yolo26_faces/weights/best.pt
Marim STRICT latimea chenarelor...

image 1/1 /content/drive/MyDrive/Team_Face_Dataset/Face_Detection.v1i.yolo26/test/images/IMG_1569-2_jpg.rf.a626835e7439c4b32a9d80f1bf542838.jpg: 640x640 2 faces, 9.7ms
Speed: 70.0ms preprocess, 9.7ms inference, 2.1ms postprocess per image at shape (1, 3, 640, 640)
[IMG_1569-2_jpg.rf.a626835e7439c4b32a9d80f1bf542838.jpg] browDownLeft (AU Score): 0.000
[IMG_1569-2_jpg.rf.a626835e7439c4b32a9d80f1bf542838.jpg] browDownRight (AU Score): 0.001
[IMG_1569-2_jpg.rf.a626835e7439c4b32a9d80f1bf542838.jpg] browInnerUp (AU Score): 0.036
[IMG_1569-2_jpg.rf.a626835e7439c4b32a9d80f1bf542838.jpg] mouthSmileLeft (AU Score): 0.000

image 1/1 /content/drive/MyDrive/Team_Face_Dataset/Face_Detection.v1i.yolo26/test/images/20260727_133551_jpg.rf.acfd286458f089a2a771fa4a4c360d30.jpg: 6

Varianta A: Detecție Raw Micro-Expresii (Blendshapes)

In [ ]:
# Testul lui Gabi
# Video

import os
import cv2
from roboflow import Roboflow
from ultralytics import YOLO
import mediapipe as mp
import urllib.request
from collections import deque
import numpy as np

# Thresholds
BUFFER_SIZE = 30
POSE_VELOCITY_THRESH = 3.0
AU_VELOCITY_THRESH = 0.15

blendshape_buffer = deque(maxlen = BUFFER_SIZE)



model_url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task"
urllib.request.urlretrieve(model_url, "face_landmarker.task")
print("Downloaded face_landmarker succesfully!")


def run_inference_test_blendshapes_video():
    best_model_path = "best.pt"
    last_pose = None

    if os.path.exists(best_model_path):
        print(f"\nModel găsit cu succes la: {best_model_path}")
        print("Marim STRICT latimea chenarelor...")
        custom_model = YOLO(best_model_path)

        BaseOptions = mp.tasks.BaseOptions
        FaceLandmarker = mp.tasks.vision.FaceLandmarker
        FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
        VisionRunningMode = mp.tasks.vision.RunningMode
        options = FaceLandmarkerOptions(
            base_options=BaseOptions(model_asset_path="face_landmarker.task"),
            running_mode=VisionRunningMode.IMAGE,
            output_face_blendshapes = True,
            output_facial_transformation_matrixes = True)
        with FaceLandmarker.create_from_options(options) as landmarker:
          cap = cv2.VideoCapture(0)

          while cap.isOpened():
            success, frame = cap.read()
            if not success:
              break
            img_h, img_w = frame.shape[:2]

            YOLO_results = custom_model.predict(source = frame, save = False, conf = 0.5, verbose = False)
            face_found = False

            for result in YOLO_results:
                boxes = result.boxes
                face_found = True
                for box in boxes:
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                    w = x2 - x1
                    h = y2 - y1
                    pad_x = w * 0.1
                    x1_nou = max(0, int(x1 - pad_x))
                    x2_nou = min(img_w, int(x2 + pad_x))
                    y2_nou = int(y2)
                    y1_nou = int(y1)
                    raport = w / h
                    if raport > 0.9:
                        offset_frunte = h * 0.30
                        y1_nou = max(0, int(y1 - offset_frunte))
                        color = (0, 0, 255)
                        label = f"Ajustat sus + latit: W/H {raport:.2f}"
                    else:
                        color = (0, 255, 0)
                        label = f"Doar latit: W/H {raport:.2f}"
                    cv2.rectangle(frame, (x1_nou, y1_nou), (x2_nou, y2_nou), color, 2)
                    text_y = max(20, y1_nou - 10)
                    cv2.putText(frame, label, (x1_nou, text_y),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
                    img_crop = frame[y1_nou:y2_nou, x1_nou:x2_nou]
                    if img_crop.size == 0:
                      print(f"Eroare: crop cu dimensiune 0!\n")
                      continue
                    img_crop_rgb = cv2.cvtColor(img_crop, cv2.COLOR_BGR2RGB)
                    mp_img = mp.Image(image_format = mp.ImageFormat.SRGB, data = img_crop_rgb)
                    MP_results = landmarker.detect(mp_img)
                    if MP_results.face_landmarks:
                      landmarks = MP_results.face_landmarks[0]
                      crop_h, crop_w, _ = img_crop.shape
                      for landmark in landmarks:
                        # Punem offset pentru a desena landmark-erii pe imaginea originala, nu pe cea cropped (e comentariul MEU nu al lui Gemini)
                        x = x1_nou + int(landmark.x * crop_w)
                        y = y1_nou + int(landmark.y * crop_h)
                        cv2.circle(frame, (x, y), 1, (0, 255, 255), -1)
                    if not MP_results.face_blendshapes:
                      continue
                    is_moving_too_fast = False
                    if MP_results.facial_transformation_matrixes:
                      matrix = MP_results.facial_transformation_matrixes[0][:3, :3]
                      euler_angles, _, _, _, _, _, = cv2.RQDecomp3x3(matrix)
                      current_pose = np.array(euler_angles) # [Pitch, Yaw, Roll]

                      if last_pose is not None:
                        pose_velocity = np.abs(current_pose - last_pose)

                        if np.any(pose_velocity > POSE_VELOCITY_THRESH):
                          is_moving_too_fast = True
                          motion_text_y = max(20, y1_nou - 15)
                          cv2.putText(frame, "MOTION GATE ACTIVE", (x1_nou, motion_text_y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                        last_pose = current_pose
                      current_aus = np.array([b.score for b in MP_results.face_blendshapes[0]])
                      au_names = [b.category_name for b in MP_results.face_blendshapes[0]]

                      if is_moving_too_fast:
                        # GATE CLOSED: Do not add corrupted data to baseline
                        pass
                      else:
                        # GATE OPEN: Safe to process
                        if len(blendshape_buffer) == BUFFER_SIZE:
                          baseline_aus = np.mean(blendshape_buffer, axis=0)
                          au_velocity = current_aus - baseline_aus

                          spike_indices = np.nonzero(au_velocity > AU_VELOCITY_THRESH)[0]

                          y_offset = y1_nou + 20
                          for idx in spike_indices:
                            if au_names[idx] == "_neutral":
                              print("Neutral Micro-expression detected!")
                              continue

                            print(f"MICRO-EXPRESSION: {au_names[idx]} (Spike: {au_velocity[idx]:.2f}")
                            cv2.putText(frame, f"Spike: {au_names[idx]}", (x2_nou +10, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                            y_offset += 25 # Move text down for multiple simultaneous spikes

                      blendshape_buffer.append(current_aus)
                    # we break the "for box in boxes:" loop so we only process the first face
                    break

            if not face_found:
             blendshape_buffer.clear()
             last_pose = None

            cv2.imshow("Micro-Expression Detector", frame)
            if (cv2.waitKey(1) & 0xFF) == 27: #Press ESC to exit
              break
        cap.release()
        cv2.destroyAllWindows()
    else:
        print(f"Eroare: Nu am găsit modelul la calea: {best_model_path}")



run_inference_test_blendshapes_video()

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Downloaded face_landmarker succesfully!

Model găsit cu succes la: /content/drive/MyDrive/Team_Face_Dataset/runs/detect/yolo26_faces/weights/best.pt
Marim STRICT latimea chenarelor...


error: OpenCV(4.10.0) /io/opencv/modules/highgui/src/window.cpp:1295: error: (-2:Unspecified error) The function is not implemented. Rebuild the library with Windows, GTK+ 2.x or Cocoa support. If you are on Ubuntu or Debian, install libgtk2.0-dev and pkg-config, then re-run cmake or configure script in function 'cvDestroyAllWindows'


Varianta B: Detecție Emoții (Ekman Mapping)

In [ ]:

# Gabi

# Acelasi lucru ca mai sus doar ca de data aceasta afisam emotii pe baza micro-expresiilor (ne folosim de dictionarul de mai jos pentru asta)
# Singurele schimbari sunt legate de loop-ul "for idx in spike_indices:"
# Am vrut sa fie o celula separata de cod pentru istoric si in caz ca vrem varianta doar cu micro-expresii

# Video

import os
import cv2
from roboflow import Roboflow
from ultralytics import YOLO
import mediapipe as mp
import urllib.request
from collections import deque
import numpy as np


import os
import cv2
from roboflow import Roboflow
from ultralytics import YOLO
import mediapipe as mp
import urllib.request
from collections import deque
import numpy as np


# Emotion Mapping (MediaPipe Blendshapes to Ekman's Universal Emotions)
# A map from blenshapes to FACS (Dr. Paul Ekman’s Facial Action Coding System)
EMOTION_MAPPING = {
    "Happiness": ["mouthSmileLeft", "mouthSmileRight", "cheekSquintLeft", "cheekSquintRight"],
    "Sadness": ["browInnerUp", "mouthFrownLeft", "mouthFrownRight"],
    "Surprise": ["jawOpen", "browOuterUpLeft", "browOuterUpRight", "eyeWidenedLeft", "eyeWidenedRight"],
    "Anger": ["browDownLeft", "browDownRight", "mouthPressLeft", "mouthPressRight"],
    "Disgust": ["noseSneerLeft", "noseSneerRight", "mouthUpperUpLeft", "mouthUpperUpRight"],
    "Fear": ["mouthStretchLeft", "mouthStretchRight", "eyeWidenedLeft", "eyeWidenedRight", "browInnerUp"],
    "Contempt": ["mouthDimpleLeft", "mouthDimpleRight", "mouthLeft", "mouthRight"]
}



# default thresholds:
# BUFFER_SIZE = 30
# POSE_VELOCITY_THRESH = 3.0
# AU_VELOCITY_THRESH = 0.15



# Thresholds
BUFFER_SIZE = 30
POSE_VELOCITY_THRESH = 3
AU_VELOCITY_THRESH = 0.15

blendshape_buffer = deque(maxlen = BUFFER_SIZE)



model_url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task"
urllib.request.urlretrieve(model_url, "face_landmarker.task")
print("Downloaded face_landmarker succesfully!")


def run_inference_test_blendshapes_video():
    best_model_path = "best.pt"
    last_pose = None

    if os.path.exists(best_model_path):
        print(f"\nModel găsit cu succes la: {best_model_path}")
        print("Marim STRICT latimea chenarelor...")
        custom_model = YOLO(best_model_path)

        BaseOptions = mp.tasks.BaseOptions
        FaceLandmarker = mp.tasks.vision.FaceLandmarker
        FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
        VisionRunningMode = mp.tasks.vision.RunningMode
        options = FaceLandmarkerOptions(
            base_options=BaseOptions(model_asset_path="face_landmarker.task"),
            running_mode=VisionRunningMode.IMAGE,
            output_face_blendshapes = True,
            output_facial_transformation_matrixes = True)
        with FaceLandmarker.create_from_options(options) as landmarker:
          cap = cv2.VideoCapture(0)

          while cap.isOpened():
            success, frame = cap.read()
            if not success:
              break
            img_h, img_w = frame.shape[:2]

            YOLO_results = custom_model.predict(source = frame, save = False, conf = 0.5, verbose = False)
            face_found = False

            for result in YOLO_results:
                boxes = result.boxes
                face_found = True
                for box in boxes:
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                    w = x2 - x1
                    h = y2 - y1
                    pad_x = w * 0.1
                    x1_nou = max(0, int(x1 - pad_x))
                    x2_nou = min(img_w, int(x2 + pad_x))
                    y2_nou = int(y2)
                    y1_nou = int(y1)
                    raport = w / h
                    if raport > 0.9:
                        offset_frunte = h * 0.30
                        y1_nou = max(0, int(y1 - offset_frunte))
                        color = (255, 0, 255)
                        label = f"Ajustat sus + latit: W/H {raport:.2f}"
                    else:
                        color = (0, 255, 0)
                        label = f"Doar latit: W/H {raport:.2f}"
                    cv2.rectangle(frame, (x1_nou, y1_nou), (x2_nou, y2_nou), color, 2)
                    text_y = max(20, y1_nou - 10)
                    cv2.putText(frame, label, (x1_nou, text_y),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
                    img_crop = frame[y1_nou:y2_nou, x1_nou:x2_nou]
                    if img_crop.size == 0:
                      print(f"Eroare: crop cu dimensiune 0!\n")
                      continue
                    img_crop_rgb = cv2.cvtColor(img_crop, cv2.COLOR_BGR2RGB)
                    mp_img = mp.Image(image_format = mp.ImageFormat.SRGB, data = img_crop_rgb)
                    MP_results = landmarker.detect(mp_img)
                    if MP_results.face_landmarks:
                      landmarks = MP_results.face_landmarks[0]
                      crop_h, crop_w, _ = img_crop.shape
                      for landmark in landmarks:
                        # Punem offset pentru a desena landmark-erii pe imaginea originala, nu pe cea cropped (e comentariul MEU nu al lui Gemini)
                        x = x1_nou + int(landmark.x * crop_w)
                        y = y1_nou + int(landmark.y * crop_h)
                        cv2.circle(frame, (x, y), 1, (0, 255, 255), -1)
                    if not MP_results.face_blendshapes:
                      continue
                    is_moving_too_fast = False
                    if MP_results.facial_transformation_matrixes:
                      matrix = MP_results.facial_transformation_matrixes[0][:3, :3]
                      euler_angles, _, _, _, _, _, = cv2.RQDecomp3x3(matrix)
                      current_pose = np.array(euler_angles) # [Pitch, Yaw, Roll]
                      # print(f"DEBUG POSE: {current_pose}")

                      if last_pose is not None:
                        pose_velocity = np.abs(current_pose - last_pose)
                        # print(f"DEBUG VELOCITY: {pose_velocity}")

                        if np.any(pose_velocity > POSE_VELOCITY_THRESH):
                          is_moving_too_fast = True
                          motion_text_y = max(15, y1_nou - 30)
                          cv2.putText(frame, "MOTION GATE ACTIVE", (x1_nou, motion_text_y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                      last_pose = current_pose
                      current_aus = np.array([b.score for b in MP_results.face_blendshapes[0]])
                      au_names = [b.category_name for b in MP_results.face_blendshapes[0]]

                      if is_moving_too_fast:
                        # GATE CLOSED: Do not add corrupted data to baseline
                        print("Head is moving too fast!")
                        pass
                      else:
                        # GATE OPEN: Safe to process
                        if len(blendshape_buffer) == BUFFER_SIZE:
                          baseline_aus = np.mean(blendshape_buffer, axis=0)
                          au_velocity = current_aus - baseline_aus

                          spike_indices = np.nonzero(au_velocity > AU_VELOCITY_THRESH)[0]

                          y_offset = y1_nou + 20

                          for idx in spike_indices:
                            if au_names[idx] == "_neutral":
                              print("Neutral Micro-expression detected!")
                              continue
                            detected_emotions = []
                            for emotion, blendshapes in EMOTION_MAPPING.items():
                               if au_names[idx] in blendshapes:
                                  detected_emotions.append(emotion)

                            if detected_emotions:
                               emotion_str = ",".join(detected_emotions)
                               print(f"MICRO-EXPRESSION: {emotion_str} (Muscle: {au_names[idx]}, Intensity: {au_velocity[idx]:.2f})")
                               cv2.putText(frame, f"{emotion_str} ({au_names[idx]})", (x2_nou + 10, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
                            else:
                              print(f"MICRO-EXPRESSION: {au_names[idx]} (Spike: {au_velocity[idx]:.2f})")
                              cv2.putText(frame, f"Spike: {au_names[idx]}", (x2_nou +10, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                            y_offset += 25 # Move text down for multiple simultaneous spikes
                      blendshape_buffer.append(current_aus)
                    # we break the "for box in boxes:" loop so we only process the first face
                    break

            if not face_found:
             blendshape_buffer.clear()
             last_pose = None

            cv2.imshow("Micro-Expression Detector", frame)
            if (cv2.waitKey(1) & 0xFF) == 27: #Press ESC to exit
              break
        cap.release()
        cv2.destroyAllWindows()
    else:
        print(f"Eroare: Nu am găsit modelul la calea: {best_model_path}")



run_inference_test_blendshapes_video()